
# Elite Dangerous Local Database
> Cached Elite Dangerous systems data

In [ ]:
#| default_exp eddb.localdb

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export

import logging, typing, sqlite3

from typing import Any, NamedTuple
from contextlib import contextmanager
from edcompanion.core import configuration


In [ ]:
from confproxy.core import init_console_logging
init_console_logging(__name__)

2025-12-19T11:41:10+0100 INFO	11424	__main__	core.py	init_console_logging	40	Installed <StreamHandler stderr (INFO)> for __main__


<Logger __main__ (INFO)>

In [ ]:
#| exporti
syslog = logging.getLogger(__name__)
syslog.info(f"Loading module {__name__}")

2025-12-19T11:41:10+0100 INFO	11424	__main__	2923348250.py	<module>	3	Loading module __main__


In [ ]:
con = sqlite3.connect(":memory:")
cur = con.execute("CREATE TABLE lang(name, first_appeared)")

# This is the named style used with executemany():
data = (
    {"name": "C", "year": 1972},
    {"name": "Fortran", "year": 1957},
    {"name": "Python", "year": 1991},
    {"name": "Go", "year": 2009},
)
cur.executemany("INSERT INTO lang VALUES(:name, :year)", data)

# This is the qmark style used in a SELECT query:
params = (1972,)

cur.execute("SELECT * FROM lang WHERE first_appeared = :year", {'year':1991})
#cur.execute("SELECT * FROM lang WHERE first_appeared = ?", params)

print(cur.fetchall())
con.close()

[('Python', 1991)]


In [ ]:
#| export
class SQLiteQueryParams(NamedTuple):
    as_param: typing.Callable
    append_param: typing.Callable   
    get_params: typing.Callable  


In [ ]:
#| export

def sqlite_query_params(log=None) -> SQLiteQueryParams:
    sql_params = {}

    def as_param(name:str):
        return f":{str(name)}"
   
    def append_param(name:str, value:Any):
        assert str(name) not in sql_params, f"Duplicate parameter {name}"
        assert len(sql_params) < 32766, "SQLite does not allow more then approx. 32k bound parameters"

        last_name = str(name)
        sql_params[last_name] = value
        return as_param(last_name)
    
    def get_params():
        return sql_params.copy()
    
    return SQLiteQueryParams(
        as_param=as_param,
        append_param=append_param if log is None else lambda p: log(append_param(p)),
        get_params=get_params
    )

In [ ]:
#| export

class SQLiteConnectionInterface(typing.NamedTuple):
    commit: typing.Callable
    close: typing.Callable
    fetch: typing.Callable
    fetchrow: typing.Callable
    execute: typing.Callable
    executemany: typing.Callable


In [ ]:
#| export
def sqllite_connection_interface(
        database: str,
    ) -> SQLiteConnectionInterface:
    connection = sqlite3.connect(database, autocommit=False)

    def _close():
        connection.close()

    def _execute(sql, params):
        cursor = connection.execute(sql, params)
        connection.commit()
        return cursor
    
    def _execute_many(sql, params):
        cursor = connection.executemany(sql, params)
        connection.commit()
        return cursor
    
    
    return SQLiteConnectionInterface(
        close=con.close,
        fetch=con.cursor().fetchall,
        fetch_as_dataframe=con.cursor().fetchdf,
        fetchrow=con.cursor().fetchone,
        execute=con.cursor().execute,
        executemany=con.cursor().executemany
)

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()